# SPIDER — Notebook 2: the image arm

Builds the learned-representation baselines to compare against the geometric
descriptor from Notebook 1, on **identical patient-stratified folds**.

Arms compared:

| arm | representation |
|---|---|
| geometry | 5 scale-invariant measurements, per-level z-scored |
| DINOv2 probe | frozen ViT-S/14 embeddings of disc crops + logistic regression |
| fine-tuned CNN | EfficientNet-B0 trained end to end on the crops |
| fusion | embeddings + geometry via gradient boosting |

**Set Runtime > Change runtime type > T4 GPU before running.**

Honest framing note: the masks that produced the geometry were themselves made
by a segmentation network. This notebook therefore compares *learned
segmentation followed by explicit measurement* against *end-to-end learned
classification* — not "measurement versus deep learning".

## Cell 1 — Setup and recover the cohort from Notebook 1

In [ ]:
import os, glob, json, time
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/spider'
os.makedirs(DRIVE, exist_ok=True)

prim = pd.read_csv(f'{DRIVE}/spider_primary_cohort.csv')
print(f"cohort: {len(prim)} discs, {prim.Patient.nunique()} patients")
print(f"levels: {sorted(prim.ivd_label.unique())}")

INV = ['disc_height_ratio', 'ap_ratio', 'canal_ratio', 'volume_ratio', 'slip_pct']
missing = [c for c in INV if c not in prim.columns]
assert not missing, f"missing descriptor columns: {missing} -- re-run Notebook 1 Cell 6"

import torch
print(f"\ntorch {torch.__version__}, GPU: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("!! no GPU -- Runtime > Change runtime type > T4 GPU, then re-run")

## Cell 2 — Download the images (3.7 GB, once per session)

Skipped automatically if a crop archive already exists on Drive.

In [ ]:
CROPS = f'{DRIVE}/spider_crops.npz'
NEED_IMAGES = not os.path.exists(CROPS)
print("crops already on Drive -- skipping the 3.7 GB download"
      if not NEED_IMAGES else "no crops yet -- downloading images")

if NEED_IMAGES:
    t0 = time.time()
    !wget -q --show-progress -O images.zip "https://zenodo.org/records/10159290/files/images.zip?download=1"
    !wget -q -O masks.zip "https://zenodo.org/records/10159290/files/masks.zip?download=1"
    !unzip -q -o images.zip -d spider_images
    !unzip -q -o masks.zip  -d spider_masks
    !pip -q install SimpleITK
    print(f"downloaded + extracted in {time.time()-t0:.0f}s")
    print(len(glob.glob('spider_images/**/*.mha', recursive=True)), "image files")

## Cell 3 — Cut one crop per disc

For each disc we take the sagittal slice where that disc is largest, crop a
square window centred on the disc and sized to 2.5x its anterior-posterior
extent, and resize to 224x224.

T2 is preferred over T1 because the gradings — Pfirrmann especially — are read
off T2. The sequence actually used is recorded per disc so it can be reported.

In [ ]:
if NEED_IMAGES:
    import SimpleITK as sitk
    from PIL import Image

    def pick_series(pat):
        """Prefer T2, then T2 SPACE, then T1."""
        for suffix in ('_t2', '_t2_SPACE', '_t1'):
            f = f'spider_images/{pat}{suffix}.mha'
            m = f'spider_masks/{pat}{suffix}.mha'
            if os.path.exists(f) and os.path.exists(m):
                return f, m, suffix[1:]
        return None, None, None

    def crop_disc(img_arr, msk_arr, spacing, label):
        """Square crop around one disc, on its largest sagittal slice."""
        lr = int(np.argmin(msk_arr.shape))          # slice axis
        hit = (msk_arr == label)
        if hit.sum() < 30:
            return None
        counts = hit.sum(axis=tuple(a for a in range(3) if a != lr))
        s = int(np.argmax(counts))
        sl_i = np.take(img_arr, s, axis=lr).astype(np.float32)
        sl_m = np.take(hit, s, axis=lr)
        if sl_m.sum() < 10:
            return None

        pl = [a for a in range(3) if a != lr]        # the two in-plane axes
        rr, cc = np.where(sl_m)
        cr, cc_ = rr.mean(), cc.mean()
        sp_r, sp_c = spacing[pl[0]], spacing[pl[1]]
        ext_mm = max((rr.max()-rr.min())*sp_r, (cc.max()-cc.min())*sp_c)
        half_r = max(int(1.25*ext_mm/sp_r), 8)
        half_c = max(int(1.25*ext_mm/sp_c), 8)

        r0, r1 = int(cr-half_r), int(cr+half_r)
        c0, c1 = int(cc_-half_c), int(cc_+half_c)
        pad = ((max(0,-r0), max(0, r1-sl_i.shape[0])),
               (max(0,-c0), max(0, c1-sl_i.shape[1])))
        sl_i = np.pad(sl_i, pad, mode='edge')
        r0, r1, c0, c1 = r0+pad[0][0], r1+pad[0][0], c0+pad[1][0], c1+pad[1][0]
        patch = sl_i[r0:r1, c0:c1]
        if patch.size == 0:
            return None

        lo, hi = np.percentile(patch, [1, 99])
        patch = np.clip((patch-lo)/(hi-lo+1e-6), 0, 1)
        return np.array(Image.fromarray((patch*255).astype(np.uint8))
                        .resize((224, 224), Image.BILINEAR))

    imgs, keep, seqs = [], [], []
    t0 = time.time()
    for i, (pat, gp) in enumerate(prim.groupby('Patient')):
        f, m, seq = pick_series(pat)
        if f is None:
            continue
        im = sitk.ReadImage(f); mk = sitk.ReadImage(m)
        ia = sitk.GetArrayFromImage(im)
        ma = sitk.GetArrayFromImage(mk).astype(np.int16)
        sx, sy, sz = im.GetSpacing(); spacing = np.array([sz, sy, sx])
        for idx, row in gp.iterrows():
            c = crop_disc(ia, ma, spacing, 200 + int(row.ivd_label))
            if c is not None:
                imgs.append(c); keep.append(idx); seqs.append(seq)
        if (i+1) % 40 == 0:
            print(f"  {i+1} patients  {time.time()-t0:.0f}s", flush=True)

    X_img = np.stack(imgs)
    np.savez_compressed(CROPS, images=X_img,
                        row_index=np.array(keep), sequence=np.array(seqs))
    print(f"\n{len(X_img)} crops from {prim.loc[keep].Patient.nunique()} patients"
          f" in {time.time()-t0:.0f}s -> saved to Drive")

z = np.load(CROPS, allow_pickle=True)
X_img, row_index, seq_used = z['images'], z['row_index'], z['sequence']
print(f"crops: {X_img.shape}, sequences used: "
      f"{pd.Series(seq_used).value_counts().to_dict()}")

# restrict the cohort to discs that actually produced a crop, keeping row order
prim = prim.loc[row_index].reset_index(drop=True)
print(f"cohort aligned to crops: {len(prim)} discs, {prim.Patient.nunique()} patients")

## Cell 4 — Look at some crops

Always eyeball the input before training on it. Each panel should show a disc
roughly filling the middle with the adjacent vertebral bodies above and below.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
pick = np.random.default_rng(0).choice(len(X_img), 12, replace=False)
for ax, i in zip(axes.ravel(), pick):
    ax.imshow(X_img[i], cmap='gray'); ax.axis('off')
    r = prim.iloc[i]
    ax.set_title(f"L{int(r.ivd_label)} n={int(r.narrowing)} P{int(r.pfirrmann)}",
                 fontsize=8)
plt.tight_layout(); plt.show()

## Cell 5 — Frozen DINOv2 embeddings

No backbone parameter is updated. With ~1,000 discs and a few hundred positives
this is the right regime for a foundation model, exactly as in the BUU study.

In [ ]:
EMB = f'{DRIVE}/spider_dinov2.npy'
if os.path.exists(EMB):
    E = np.load(EMB)
    print(f"embeddings loaded from Drive: {E.shape}")
else:
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(dev).eval()

    mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

    outs, B = [], 64
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, len(X_img), B):
            b = torch.from_numpy(X_img[i:i+B]).float().to(dev) / 255.
            b = b.unsqueeze(1).repeat(1, 3, 1, 1)          # grey -> 3 channels
            b = (b - mean) / std
            outs.append(dino(b).cpu().numpy())
            if i % (B*8) == 0:
                print(f"  {i}/{len(X_img)}  {time.time()-t0:.0f}s", flush=True)
    E = np.concatenate(outs)
    np.save(EMB, E)
    print(f"embeddings: {E.shape} in {time.time()-t0:.0f}s -> saved to Drive")

## Cell 6 — Fold-matched evaluation

Every arm sees the identical `GroupKFold` split on patient. Per-level
z-scoring, PCA and standardisation are all fitted on training folds only.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SEED, N_SPLITS, N_PCA = 0, 5, 128
cv = GroupKFold(n_splits=N_SPLITS)
groups = prim['Patient'].values
levels = prim['ivd_label'].values
FOLDS = list(cv.split(np.zeros(len(prim)), np.zeros(len(prim)), groups))
print(f"{N_SPLITS} folds, test sizes: {[len(te) for _, te in FOLDS]}")


def _zscore(ref, ref_lvl, X, lvl):
    out = X.copy()
    gm, gs = ref.mean(), ref.std().replace(0, 1)
    for l in np.unique(lvl):
        m = ref_lvl == l
        mu = ref[m].mean() if m.sum() >= 20 else gm
        sd = ref[m].std().replace(0, 1) if m.sum() >= 20 else gs
        out.loc[lvl == l] = (X.loc[lvl == l] - mu) / sd
    return out


def lr():
    return LogisticRegression(max_iter=5000, class_weight='balanced',
                              random_state=SEED)


def run_geometry(y):
    p = np.zeros(len(y))
    for tr, te in FOLDS:
        A, B = prim[INV].iloc[tr], prim[INV].iloc[te]
        A2 = _zscore(A, levels[tr], A, levels[tr])
        B2 = _zscore(A, levels[tr], B, levels[te])
        clf = make_pipeline(SimpleImputer(strategy='median'),
                            StandardScaler(), lr()).fit(A2, y[tr])
        p[te] = clf.predict_proba(B2)[:, 1]
    return p


def run_embeddings(y):
    p = np.zeros(len(y))
    for tr, te in FOLDS:
        clf = make_pipeline(StandardScaler(),
                            PCA(n_components=min(N_PCA, len(tr)-1),
                                random_state=SEED),
                            lr()).fit(E[tr], y[tr])
        p[te] = clf.predict_proba(E[te])[:, 1]
    return p


def run_fusion(y):
    from sklearn.ensemble import HistGradientBoostingClassifier
    p = np.zeros(len(y))
    for tr, te in FOLDS:
        A, B = prim[INV].iloc[tr], prim[INV].iloc[te]
        A2 = _zscore(A, levels[tr], A, levels[tr]).values
        B2 = _zscore(A, levels[tr], B, levels[te]).values
        pca = make_pipeline(StandardScaler(),
                            PCA(n_components=min(32, len(tr)-1),
                                random_state=SEED)).fit(E[tr])
        Atr = np.hstack([A2, pca.transform(E[tr])])
        Ate = np.hstack([B2, pca.transform(E[te])])
        w = np.where(y[tr] == 1, (y[tr] == 0).sum()/max((y[tr] == 1).sum(), 1), 1.)
        m = HistGradientBoostingClassifier(max_depth=3, max_iter=300,
                                           random_state=SEED)
        m.fit(Atr, y[tr], sample_weight=w)
        p[te] = m.predict_proba(Ate)[:, 1]
    return p


TARGETS = {'disc narrowing': 'narrowing', 'disc bulging': 'bulging',
           'Pfirrmann >= 4': 'pfirrmann_severe', 'Modic (any)': 'modic_any'}
ARMS = {'geometry (5, z)': run_geometry,
        'DINOv2 probe': run_embeddings,
        'fusion (geom+emb)': run_fusion}

oof, rows = {}, []
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for aname, fn in ARMS.items():
        t0 = time.time()
        p = fn(y)
        oof[(tname, aname)] = p
        rows.append(dict(target=tname, arm=aname, auc=roc_auc_score(y, p)))
        print(f"  {tname:16s} {aname:20s} {rows[-1]['auc']:.3f}"
              f"  ({time.time()-t0:.0f}s)", flush=True)

res = pd.DataFrame(rows)
print("\nPooled out-of-fold AUC, identical folds")
print("=" * 60)
print(res.pivot(index='target', columns='arm', values='auc').round(3).to_string())

## Cell 7 — Significance against the geometric descriptor

Patient-level bootstrap, 2,000 resamples — the same convention as the
BUU-LSPINE study, so the two papers' tables are directly comparable.

In [ ]:
N_BOOT = 2000
rng = np.random.default_rng(SEED)
pats, uniq = prim['Patient'].values, prim['Patient'].unique()
boot_idx = []
for _ in range(N_BOOT):
    pick = rng.choice(uniq, len(uniq), replace=True)
    boot_idx.append(np.concatenate([np.flatnonzero(pats == u) for u in pick]))

print("dAUC vs geometry  [95% CI]   (negative = worse than geometry)")
print("=" * 66)
out = []
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    base = oof[(tname, 'geometry (5, z)')]
    print(f"\n{tname}")
    for aname in ARMS:
        if aname == 'geometry (5, z)':
            continue
        p = oof[(tname, aname)]
        d = [roc_auc_score(y[i], p[i]) - roc_auc_score(y[i], base[i])
             for i in boot_idx if len(np.unique(y[i])) > 1]
        lo, hi = np.percentile(d, [2.5, 97.5])
        verdict = ('geometry better' if hi < 0 else
                   'this arm better' if lo > 0 else 'indistinguishable')
        print(f"   {aname:20s} {np.mean(d):+.3f}  [{lo:+.3f}, {hi:+.3f}]  {verdict}")
        out.append(dict(target=tname, arm=aname, dauc=np.mean(d),
                        lo=lo, hi=hi, verdict=verdict))

pd.DataFrame(out).to_csv(f'{DRIVE}/notebook2_significance.csv', index=False)
res.to_csv(f'{DRIVE}/notebook2_auc.csv', index=False)
print("\nsaved to Drive")

## Cell 8 — Optional: fine-tuned EfficientNet-B0

Slowest arm — roughly 15–25 minutes on a T4 for one target. Run it for the
primary target only unless you have time. Frozen embeddings are usually the
stronger image pathway at this sample size, so this arm mainly establishes that
the comparison was not rigged in the geometry's favour.

In [ ]:
RUN_CNN = False        # set True when you have the GPU time
CNN_TARGET = 'narrowing'

if RUN_CNN:
    import torch, torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    import torchvision
    dev = 'cuda'
    y = prim[CNN_TARGET].astype(int).values
    p_cnn = np.zeros(len(y))

    for k, (tr, te) in enumerate(FOLDS):
        net = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, 1)
        net = net.to(dev)
        Xt = torch.from_numpy(X_img).float().div(255).unsqueeze(1).repeat(1,3,1,1)
        dl = DataLoader(TensorDataset(Xt[tr], torch.tensor(y[tr]).float()),
                        batch_size=32, shuffle=True)
        pw = torch.tensor([(y[tr]==0).sum()/max((y[tr]==1).sum(),1)]).to(dev)
        lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
        opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
        net.train()
        for ep in range(6):
            for xb, yb in dl:
                opt.zero_grad()
                loss = lossf(net(xb.to(dev)).squeeze(1), yb.to(dev))
                loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            pr = []
            for i in range(0, len(te), 64):
                pr.append(torch.sigmoid(
                    net(Xt[te[i:i+64]].to(dev)).squeeze(1)).cpu().numpy())
        p_cnn[te] = np.concatenate(pr)
        print(f"  fold {k+1}/{N_SPLITS} done", flush=True)

    print(f"\nfine-tuned CNN, {CNN_TARGET}: AUC {roc_auc_score(y, p_cnn):.3f}")
    np.save(f'{DRIVE}/oof_cnn_{CNN_TARGET}.npy', p_cnn)